# Silver Layer - Limpieza y Transformación

Transforma los datos crudos de `e_commerce.bronze.bronze_pedidos_raw` hacia la tabla Delta `e_commerce.silver.ventas_limpias` (una fila por producto comprado), aplicando limpieza y normalización de tipos.

## Transformaciones aplicadas

* **Parseo de `cliente_info`**: separación en `nombre_cliente`, `email` y `ciudad` (split por `|`)
* **Normalización de montos**: eliminación de símbolos de moneda (`USD`, `$`, `ARS`) y conversión de separadores de miles/decimales, con `try_cast` a float
* **Normalización de `estado_pago`**: lowercase + trim, con valor por defecto `desconocido`
* **Dirección de envío**: nulos y vacíos → `Desconocida`
* **Explosión de productos**: `productos_comprados` se divide en ítems individuales (una fila por producto), extrayendo `nombre_producto` y `cantidad_producto` desde el patrón `(xN)`
* **Deduplicación**: `dropDuplicates()` sobre el resultado final

In [0]:
from pyspark.sql.functions import col, explode, lower, regexp_replace, split, trim, when, expr, lit, coalesce

In [0]:
df_bronze = spark.table("e_commerce.bronze.bronze_pedidos_raw")

In [0]:
df_silver_base = (
    df_bronze
    .withColumn("id_transaccion", trim(col("id_transaccion")))
    .withColumn("fecha_operacion", coalesce(col("fecha_operacion"), lit("fecha_desconocida")))
    .withColumn("nombre_cliente", trim(split(col("cliente_info"), r"\|")[0]))
    .withColumn("email", trim(split(col("cliente_info"), r"\|")[1]))
    .withColumn("ciudad", trim(split(col("cliente_info"), r"\|")[2]))
    .withColumn(
        "monto_limpio",
        regexp_replace(col("monto_total"), r"[A-Za-z\$\s]", "")
    )
    .withColumn(
        "monto_normalizado",
        regexp_replace(regexp_replace(col("monto_limpio"), r"\.", ""), r",", ".")
    )
    .withColumn(
        "monto_total",
        expr("try_cast(monto_normalizado as float)")
    )
    .withColumn("estado_pago", coalesce(lower(trim(col("estado_pago"))), lit("desconocido")))
    .withColumn(
        "direccion_envio",
        when(
            (col("direccion_envio") == "") | col("direccion_envio").isNull(),
            "Desconocida",
        ).otherwise(trim(col("direccion_envio")))
    )
)

In [0]:
df_productos_unificados = df_silver_base.withColumn(
    "productos_array",
    split(regexp_replace(col("productos_comprados"), " / ", ","), ","),
)

df_silver_exploded = df_productos_unificados.withColumn(
    "producto_item", explode(col("productos_array"))
)

In [0]:
df_silver_final = (
    df_silver_exploded.withColumn(
        "nombre_producto", trim(regexp_replace(col("producto_item"), "\\s*\\(x\\d+\\)", ""))
    )
    .withColumn(
        "cantidad_producto",
        regexp_replace(
            regexp_replace(col("producto_item"), ".*\\(x", ""), "\\)", ""
        ).cast("int"),
    )
    .select(
        "id_transaccion",
        "fecha_operacion", 
        "nombre_cliente",
        "email",
        "ciudad",
        "nombre_producto",
        "cantidad_producto",
        "monto_total",
        "estado_pago",
        "direccion_envio",
    )
    .dropDuplicates()
)

In [0]:
df_silver_final.write.format("delta").mode("overwrite").saveAsTable(
    "e_commerce.silver.ventas_limpias"
)